In [ ]:
#引入包
import torch
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import sys
import torchvision.transforms 

#超参数定义
data_root = "./data"
img_path = "./data/img"
mask_path = "./data/mask"
batch = 20

#自定义数据类
class MyDataset(Dataset):
    def __init__(self, data_root):
        super(MyDataset, self).__init__()

        #检查路径
        if not os.path.exists(data_root):
            raise FileNotFoundError(f"数据根目录不存在：{data_root}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"图像文件夹不存在：{img_path}")
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"掩码文件夹不存在：{mask_path}")
            
        self.img_files = sorted(os.listdir(self.img_dir), 
                                 key=lambda x: int(x.split('.')[0]))
        self.mask_files = sorted(os.listdir(self.mask_dir), 
                                 key=lambda x: int(x.split('.')[0]))
        self.transform = torchvision.transforms.ToTensor()
    def __len__(self):
        return len(self.img_files)
    def __getitem__(self, idx):
        transforms = torchvision.transforms.ToTensor()
        img = transforms(Image.open(os.path.join(img_path, f"{idx}.png")).convert("L"))#转换单通道、张量、归一化
        mask = transforms(Image.open(os.path.join(mask_path, f"{idx}.png")).convert("L"))#转换单通道、张量、归一化
        return self.transform(img), self.transform(mask)

data = MyDataset(data_root)#实例化数据集

#数据迭代器
dataloader = DataLoader(dataset=data, batch_size=batch, shuffle=True, drop_last=False)

print(type(dataloader))

#残差卷积块定义
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        #主路径 Main Path
        self.mp = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3,stride=1, padding=1, bias=False)
            nn.BatchNorm2d(out_channels)
            nn.ReLU(inplace=True)
                
            nn.Conv2d(out_channels, out_channels, kernel_size=3,stride=1, padding=1, bias=False)
            nn.BatchNorm2d(out_channels)
            nn.ReLU(inplace=True)
        )
        # 捷径路径 (Shortcut / Skip Connection)
        # 如果输入输出维度不一致（通道数变了或尺寸变了），需要用 1x1 卷积调整 identity
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        identity = x
        conv = self.mp(x)
        identity = self.shortcut(identity)
        out += identity
        out = self.relu(out)
        return out

#Attention Gate Block
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True)
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 使用残差块提取特征
        self.res_block = ResBlock(in_channels, out_channels)
        # 下采样：长宽减半
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        f = self.res_block(x) # 提取后的特征（用于跳跃连接）
        p = self.pool(f)      # 下采样后的特征（传给下一层）
        return f, p

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DecoderBlock, self).__init__()
        # 上采样：长宽翻倍
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # 注意力门：F_g 是上层传上来的，F_l 是左边跳过来的
        # 通常上采样后通道数和左边跳过来的一致，所以 F_int 设为 out_ch
        self.att_gate = AttentionGate(F_g=in_ch, F_l=out_ch, F_int=out_ch // 2)
        
        # 拼接后的残差卷积块：in_ch (来自上层) + out_ch (来自跳跃连接)
        self.res_block = ResidualBlock(in_ch + out_ch, out_ch)

    def forward(self, x, skip):
        """
        x: 来自深层的特征图 (Gate 信号)
        skip: 来自编码层同层的特征图 (Local 信号)
        """
        # 1. 上采样
        g = self.upsample(x)
        
        # 2. 注意力过滤：用上采样的信号 g 过滤 skip
        s = self.att_gate(g=g, x=skip)
        
        # 3. 拼接并过残差块
        d = torch.cat([s, g], dim=1)
        return self.res_block(d)

class AttResUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1): # <--- 这里改成了1，匹配你的单通道图片
        super(AttResUNet, self).__init__()
        
        # Encoder (左半部分)
        self.enc1 = EncoderBlock(in_ch, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        
        # Bottleneck (最底层)
        self.bottleneck = ResidualBlock(256, 512)
        
        # Decoder (右半部分)
        # 每一层 DecoderBlock 内部都集成了 AttentionGate 和 ResidualBlock
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)
        
        # 最终输出层：将64通道映射回1个通道（掩码）
        self.final = nn.Conv2d(64, out_ch, kernel_size=1)
        # 如果是二分类分割，通常最后接一个 Sigmoid
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Encoder (下采样过程)
        # f 是用于跳跃连接的特征，p 是池化后传给下一层的特征
        f1, p1 = self.enc1(x)
        f2, p2 = self.enc2(p1)
        f3, p3 = self.enc3(p2)
        
        # 2. Bottleneck
        b = self.bottleneck(p3)
        
        # 3. Decoder (上采样 + 注意力融合)
        # 注意：DecoderBlock 内部自动处理了 [上采样 -> Attention -> Concat -> Conv]
        d3 = self.dec3(b, f3)
        d2 = self.dec2(d3, f2)
        d1 = self.dec1(d2, f1)
        
        out = self.final(d1)
        return self.sigmoid(out)

model = AttResUNet(in_ch=1, out_ch=1).to(device)
criterion = nn.BCELoss() # 二分类常用的交叉熵
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for images, masks in dataloader:
    images = images.to(device) # [batch, 1, H, W]
    masks = masks.to(device)   # [batch, 1, H, W]
    
    # 前向传播
    outputs = model(images)
    loss = criterion(outputs, masks)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()